In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Perturb.datamodule import PerturberDataModule
from T_perturb.Perturb.trainer import PerturberTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed_no = 42
pl.seed_everything(seed_no)
torch.manual_seed(seed_no)

Seed set to 42


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70]
perturbation_token = 1


In [5]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [6]:
type(src_dataset)

datasets.arrow_dataset.Dataset

In [7]:
conditions = None
condition_keys = None
conditions_combined = None

In [8]:
if condition_keys is None:
    condition_keys = 'tmp_batch'
    # create a mock vector if there are no batch effect
    tmp_series = pd.DataFrame(
        {
            condition_keys: np.ones(num_samples),
        }
    )


In [9]:
if isinstance(condition_keys, str):
    condition_keys_ = [condition_keys]
else:
    condition_keys_ = condition_keys

if conditions is None:
    if condition_keys is not None:
        conditions_ = {}
        for cond in condition_keys_:
            conditions_[cond] = tmp_series[cond].unique().tolist()
    else:
        conditions_ = {}
else:
    conditions_ = conditions

if conditions_combined is None:
    if len(condition_keys_) > 1:
        tmp_series['conditions_combined'] = tmp_series[condition_keys].apply(
            lambda x: '_'.join(x), axis=1
        )
    else:
        tmp_series['conditions_combined'] = tmp_series[condition_keys]
    conditions_combined_ = tmp_series['conditions_combined'].unique().tolist()
else:
    conditions_combined_ = conditions_combined

condition_encodings = {
    cond: {k: v for k, v in zip(conditions_[cond], range(len(conditions_[cond])))}
    for cond in conditions_.keys()
}
conditions_combined_encodings = {
    k: v for k, v in zip(conditions_combined_, range(len(conditions_combined_)))
}

In [10]:
tgt_adata_tmp = sc.AnnData(X=tgt_counts_dict['tgt_h5ad_t1'].squeeze(), obs=tmp_series)

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [11]:
if (condition_encodings is not None) and (condition_keys_ is not None):
    conditions = [
        label_encoder(
            tgt_adata_tmp,
            encoder=condition_encodings[condition_keys_[i]],
            condition_key=condition_keys_[i],
        )
        for i in range(len(condition_encodings))
    ]
    conditions = torch.tensor(conditions, dtype=torch.long).T
    conditions_combined = label_encoder(
        tgt_adata_tmp,
        encoder=conditions_combined_encodings,
        condition_key='conditions_combined',
    )
    conditions_combined = torch.tensor(conditions_combined, dtype=torch.long)


In [12]:
# check if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [13]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.Perturb.datamodule)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberTrainer
from T_perturb.Perturb.datamodule import PerturberDataModule

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    'perturbation_token': perturbation_token,
    'context_mode': True,
    'perturbation_mode': 'generate',
    'perturbation_sequence': ['src','tgt'],
    'temperature':1.5,
    'iterations': 19,
    'sequence_length': max_seq_length - 10,
    'pos_encoding_mode': 'time_pos_sin',
    'return_attn': False,
    'mask_scheduler': 'cosine'
}
decoder_module = PerturberTrainer(
    **trainer_params
)

celltype_to_perturb = 'A'
data_module = PerturberDataModule(
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    batch_size=batch_size,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
    condition_keys=condition_keys_,
    condition_encodings=condition_encodings,
    conditions=conditions,
    conditions_combined=conditions_combined,
    celltype_to_perturb=celltype_to_perturb,
    filter_mode='include'
)
data_module.setup()
test_loader = data_module.test_dataloader()

cls_token_1 tensor([101])
cls_token_2 tensor([102])
Start datamodule
src_dataset_filtered Dataset({
    features: ['input_ids', 'cell_type', 'length', 'cell_pairing_index'],
    num_rows: 32
})
[100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
src_len 32
tgt_len 43


/lustre/scratch126/cellgen/team361/kl11/t_generative/T_perturb/T_perturb/Dataloaders/datamodule.py:62: UserWarning: src and tgt dataset have different length
  warn('src and tgt dataset have different length')


In [14]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
    accelerator=accelerator,
    devices=1 if torch.cuda.is_available() else 0,  # inference only on one gpu
)
trainer.test(
    decoder_module, 
    data_module,
    ckpt_path='T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt',
    )


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.
Restoring states from the checkpoint path at T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


src_dataset_filtered Dataset({
    features: ['input_ids', 'cell_type', 'length', 'cell_pairing_index'],
    num_rows: 32
})
[100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
src_len 32
tgt_len 43


Loaded model weights from the checkpoint at T_perturb/T_perturb/tests/checkpoints/test_masking_checkpoint-epoch=00.ckpt
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]perturbating tgt
perturbation cuda:0
perturbed_tgt cuda:0
perturbating tgt
perturbation cuda:0
perturbed_tgt cuda:0
perturbating src
tensor([[ 76,  36,  58,  45,   9,   0,  63,  14,  27,  99,  46,  86,  81,   1,
           6,  32,  94,   1,  31,   7,  12,  96,  21,  61,  92,  42,  97,  60,
          43,  69,  84,  22,  65,  38,  73,   8,  59,  44,  71,  37,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [ 21,  94,  55,  63,  62,  34,  83,  20,  77,  87,  98,  82,   4,  35,
          67,  59,  11,  89,  26,  14,  58,  30,  85,   1,  73,  45,  93,  69,
          38,  31,  57, 100,  39,  22,  90,   6,  28,  46,  40,  44,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [ 94,  24,  96,  33,  82,  61,  90,  19,  21,   0,  71,  54,  27,  55,
          73,  50,  18,   5,   6,  65,  83,  15,  93,   4,  81,  49,  22,  35,
           3,  31,  75,  67,  87,  76,  32,  13,  79,  51,   7,   1,   0,   

100%|██████████| 19/19 [00:00<00:00, 42.70it/s]


...

100%|██████████| 19/19 [00:00<00:00, 84.56it/s]


pred_tps [1, 2]


100%|██████████| 19/19 [00:00<00:00, 106.72it/s]


test_dict {'true_counts': [], 'cls_embeddings': [], 'cosine_similarities': [], 'batch': [], 'cell_idx': [], 'gene_embeddings_t1': [], 'gene_embeddings_t2': [], 'cls_cosine_similarity': [], 'mean_cosine_similarity': [], 'delta_probs': [], 'rouge1_25': [[0.6, 0.6122448979591836, 0.48, 0.4897959183673469]], 'rouge1_100': [[0.7532467532467534, 0.7012987012987013, 0.7792207792207793, 0.7012987012987013]], 'rouge1_60': [[0.7532467532467534, 0.7012987012987013, 0.7792207792207793, 0.7012987012987013]]}
test_dict {'true_counts': [], 'cls_embeddings': [], 'cosine_similarities': [], 'batch': [], 'cell_idx': [], 'gene_embeddings_t1': [], 'gene_embeddings_t2': [], 'cls_cosine_similarity': [], 'mean_cosine_similarity': [], 'delta_probs': [], 'rouge1_25': [[0.6, 0.6122448979591836, 0.48, 0.4897959183673469], [0.56, 0.56, 0.4897959183673469, 0.5]], 'rouge1_100': [[0.7532467532467534, 0.7012987012987013, 0.7792207792207793, 0.7012987012987013], [0.8311688311688312, 0.7532467532467534, 0.70129870129870

RuntimeError: No active exception to reraise